# Capstone — Warehouse Extension (mirrors your deployed research paper)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sanaullah-Turab/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

**Lane 2 — Refresh / Content Opportunity Scoring.** The weekly assignments (`w01`–`w07`) were
built and validated on the anonymized starter sample (`content_refresh_anonymized.csv`), as
each week's card required. This capstone extends the same lane, the same overall methodology
(baseline → model → honest grouped validation → action playbook), and the same central finding
(random-split scores overstate real performance) to the **full warehouse release** on Hugging
Face — using the data contract already defined in `w03_data_contract.ipynb` as its foundation.

> Note on scope: `w03` validated its contract on a small 5-feature honest quick-score
> (ROC-AUC only, no baseline comparison, no grouped-split audit). This notebook builds that
> contract out into a full lane deliverable: a real baseline rule, a compared model, a
> client-grouped validation audit, and a ranked action playbook — the same bar Weeks 4–7 met
> on the starter sample, now met on the warehouse.

> **Before running:** verify the exact column names below (`gsc_impressions`, `gsc_clicks`,
> `gsc_avg_position`, `ga4_data_available`, and any `dim_content`/`dim_clients` columns used)
> against `docs/data-dictionary.md` — this notebook was drafted from the schema shown in
> `w03_data_contract.ipynb`'s query cells, not from a live connection to the gated dataset.


## 1. Question

**Research question:** Given a client's content inventory, which pages should a content
reviewer look at first this week?

**Unit of analysis:** one page (`content_hash_id`), scored and ranked within a client
(`client_hash_id`), using a mid-panel month of warehouse history.

**Decision this supports:** how a reviewer with a fixed number of hours spends them across a
large backlog of pages, instead of ad hoc judgment or a single fixed rule.

**Who acts, and what they do:** a content reviewer or client-side SEO owner opens the top of
the ranked queue, reads the reason code(s) attached to each page, and takes one of four
actions: `protect`, `refresh_priority`, `refresh_review`, or `monitor`.

**Cost of a wrong call:** a false positive wastes reviewer hours on a healthy page — cheap. A
false negative lets a real declining page keep losing impressions unnoticed until the next
scoring pass — the more expensive error, because review capacity is fixed. **Precision@K** is
therefore the primary metric, with recall on the declining set watched as a secondary check.

**Why this extension matters:** the starter-sample capstone (30k rows, 32 clients) is useful,
but it's a fixed anonymized snapshot. The warehouse (~79M daily rows, full client panel) is
where this lane's real production data lives — and it's the dataset the capstone's own Data
section points to. This notebook answers the same question at that scale, using the contract
already defined in `w03`.


## 2. Data

**Release used:** `FlyRank/internship-warehouse` on Hugging Face (gated, read-token access),
via DuckDB over `hf://`. **Table:** `fact_content_daily_performance`, partition
`month=2026-03` — a single mid-panel month, per the assignment's instruction to iterate on a
mid-panel month and leave the sealed final month (`_sample`, June 2026) untouched. Joined where
needed to `dim_content` (content metadata) and `dim_clients` (`gsc_data_start` /
`ga4_data_start`, to check each client's tracking coverage before trusting a row).

**Unit of analysis (verified in `w03`):** one raw row = one `(report_date, client_hash_id,
content_hash_id)` daily performance record — confirmed duplicate-free in `w03`'s grain check.
The working feature-row here is a coarser grain defined on top of that: one row per
`(client_hash_id, content_hash_id)`, aggregated over the first half of the month.

**Label / proxy — carried forward from `w03` unchanged:** `declining_second_half` — did a
page's GSC impressions in the second half of the month (days 16–31) come in lower than the
first half (days 1–15)? This is an explicit **proxy**, not a real future-window label (see
Section 5) — the same honesty standard as the starter sample's `is_declining_label`.

**What's deliberately excluded, and why (from `w03`'s field-bucket contract):**
- Any GA4 engagement/session column on rows where `ga4_data_available IS NOT TRUE` — rows
  before a client's GA4 tracking started aren't real zero-engagement observations, and
  treating them as zeros would inject a fake signal.
- `client_hash_id` / `content_hash_id` — pseudonymous join/grouping keys, never features.
- Second-half `gsc_impressions` as a feature — it's literally what the label is computed from
  (this is the deliberate leak `w03` planted and removed; re-verified here in Section 3).
- No client names, domains, URLs, titles, or raw queries — the warehouse ships pre-anonymized
  and hashed.


In [ ]:
%pip -q install duckdb
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'  # same mid-panel month as w03_data_contract.ipynb -- keep these in sync
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

print('connected using month partition:', MONTH)


In [ ]:
# Sanity check: reproduce w03's grain + coverage numbers so this notebook stands on its own
span = con.sql(f"SELECT COUNT(*) AS n_rows, COUNT(DISTINCT client_hash_id) AS n_clients, "
               f"COUNT(DISTINCT content_hash_id) AS n_content_items, MIN(report_date) AS min_date, "
               f"MAX(report_date) AS max_date FROM {FACT_MONTH}").df()

print(f"rows in month={MONTH}: {span['n_rows'][0]:,}")
print(f"distinct clients: {span['n_clients'][0]:,} | distinct content items: {span['n_content_items'][0]:,}")
print(f"date span: {span['min_date'][0]} to {span['max_date'][0]}")

availability = con.sql(f"SELECT COUNT(*) AS total_rows, "
                        f"COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows "
                        f"FROM {FACT_MONTH}").df()
total, survive = availability['total_rows'][0], availability['ga4_available_rows'][0]
print(f"\nrows with GA4 tracking already live: {survive:,} / {total:,} ({survive/total:.1%})")


## 3. Methodology

### 3.1 Feature vector — first half of March only

Every feature is built from `report_date <= 2026-03-15`, so it's knowable at the mid-month
decision checkpoint, before the second-half outcome exists. GA4 engagement features are added
here, beyond `w03`'s original 5-feature honest quick-score, gated the same way: only aggregated
over rows where `ga4_data_available IS TRUE`, with a `has_ga4_data` flag carrying the
missingness information explicitly rather than silently zero-filling it.

### 3.2 Baseline — the same rule shape as Week 4, rebuilt on warehouse features

**Rule:** a page is worth a refresh review if it's visible (`imp_first_half` above a floor),
hasn't shown consistent activity, and its first-half CTR underperforms its own position-tier
baseline (position tiers bucketed from `avg_position_first_half`, the same weighted-baseline
logic as the starter-sample rule — CTR falls in step with position, so a flat threshold would
mislabel low-position pages as broken).

### 3.3 Model — same method sequence as Week 5

Logistic Regression (readable baseline) → Random Forest (300 trees, `max_depth=10`,
`class_weight="balanced"`, `random_state=42`). **Primary metric:** precision@50. **Leakage
guard:** `imp_second_half` and anything derived from it are never features — verified with the
same deliberate leak-and-remove test `w03` already ran.

### 3.4 Validation — client-grouped split, before/after

Same central finding as the starter-sample capstone: a naive row-level random split lets the
model see multiple pages from the same client in both train and test, inflating the score with
client recognition rather than a page-level pattern. This notebook reports **both** the naive
random-split number and the honest client-grouped number, on the same warehouse data, so the
gap itself is measured here rather than assumed to transfer from the starter-sample result.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42
np.random.seed(SEED)

# feature frame: first half of March, GSC + gated GA4 aggregates
feature_frame = con.sql(f'''
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_first_half,
        SUM(gsc_clicks) AS clk_first_half,
        AVG(gsc_avg_position) AS avg_position_first_half,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days_first_half,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE NULL END AS ctr_first_half,
        SUM(ga4_sessions) FILTER (WHERE ga4_data_available IS TRUE) AS sessions_first_half,
        AVG(ga4_engagement_rate) FILTER (WHERE ga4_data_available IS TRUE) AS engagement_rate_first_half,
        MAX(ga4_data_available IS TRUE) AS has_ga4_data
    FROM {FACT_MONTH}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 10
''').df()

feature_frame["has_ga4_data"] = feature_frame["has_ga4_data"].fillna(False).astype(int)
feature_frame["sessions_first_half"] = feature_frame["sessions_first_half"].fillna(0)
feature_frame["engagement_rate_first_half"] = feature_frame["engagement_rate_first_half"].fillna(0)

def position_tier(p):
    if p <= 0 or pd.isna(p): return "unknown"
    if p <= 3: return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    if p <= 50: return "page_3_5"
    return "deep"

feature_frame["position_tier"] = feature_frame["avg_position_first_half"].apply(position_tier)

print(f"{len(feature_frame):,} content items with enough first-half volume to score")
print(f"GA4 coverage among these: {feature_frame['has_ga4_data'].mean():.1%}")
feature_frame.head()


In [ ]:
# label: strictly second-half only, the outcome window features never touched
label_frame = con.sql(f'''
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_second_half
    FROM {FACT_MONTH}
    WHERE report_date > DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
''').df()

data = feature_frame.merge(label_frame, on=["client_hash_id", "content_hash_id"], how="inner")
data["declining_second_half"] = (data["imp_second_half"] < data["imp_first_half"]).astype(int)

print(f"{len(data):,} rows with both halves present")
print(f"base rate of declining_second_half: {data['declining_second_half'].mean():.1%}")


In [ ]:
# leakage re-verification (same trap w03 already ran, confirmed here on the full feature set)
NUMERIC_FEATURES = ["imp_first_half", "clk_first_half", "avg_position_first_half",
                     "active_days_first_half", "ctr_first_half",
                     "sessions_first_half", "engagement_rate_first_half", "has_ga4_data"]
CATEGORICAL_FEATURES = ["position_tier"]
TARGET = "declining_second_half"

model_data = data.dropna(subset=NUMERIC_FEATURES + [TARGET]).copy()

leak_check = [c for c in ["imp_second_half"] if c in NUMERIC_FEATURES + CATEGORICAL_FEATURES]
assert leak_check == [], f"Leaky columns found: {leak_check}"
print("Leakage check: PASSED -- imp_second_half is not a feature")

# deliberate injection, prove the harness still catches it (mirrors w03 section 3e)
leaky_data = model_data.copy()
leaky_data["leaked_imp_second_half"] = data.loc[model_data.index, "imp_second_half"]
X_leak = pd.get_dummies(leaky_data[NUMERIC_FEATURES + ["leaked_imp_second_half"] + CATEGORICAL_FEATURES])
y_leak = leaky_data[TARGET]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leak, y_leak, test_size=0.25, random_state=SEED, stratify=y_leak)
leaky_auc = roc_auc_score(y_te_l, LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l).predict_proba(X_te_l)[:, 1])
print(f"Leaky-feature injection test: ROC-AUC = {leaky_auc:.3f} (near 1.0 confirms the harness detects leakage)")


In [ ]:
# client-grouped vs random split, on the real (non-leaky) feature set
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": y_true.values, "score": scores})
    top = frame.sort_values("score", ascending=False).head(k)
    return float(top["y"].mean()) if len(top) else 0.0

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL_FEATURES),
])

def fit_eval(train_df, test_df, label):
    X_train = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y_train = train_df[TARGET]
    X_test  = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y_test  = test_df[TARGET]
    rf = Pipeline([
        ("pre", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=300, max_depth=10,
                                        class_weight="balanced", random_state=SEED, n_jobs=-1)),
    ])
    rf.fit(X_train, y_train)
    proba = rf.predict_proba(X_test)[:, 1]
    p20, p50 = precision_at_k(y_test, proba, 20), precision_at_k(y_test, proba, 50)
    auc = roc_auc_score(y_test, proba)
    print(f"{label:32s} n_test={len(test_df):6,}  P@20={p20:.3f}  P@50={p50:.3f}  ROC-AUC={auc:.3f}  base_rate={y_test.mean():.3f}")
    return {"label": label, "P@20": p20, "P@50": p50, "ROC-AUC": auc, "base_rate": float(y_test.mean())}

train_rand, test_rand = train_test_split(model_data, test_size=0.2, random_state=SEED, stratify=model_data[TARGET])
before = fit_eval(train_rand, test_rand, "BEFORE: random split")

clients = sorted(model_data["client_hash_id"].unique())
test_clients = set(clients[::5])
train_grp = model_data[~model_data["client_hash_id"].isin(test_clients)].copy()
test_grp  = model_data[model_data["client_hash_id"].isin(test_clients)].copy()
after = fit_eval(train_grp, test_grp, "AFTER: client-grouped split")

print(f"\nGap from switching to the honest split: P@50 drops by {before['P@50'] - after['P@50']:.2f}.")


In [ ]:
# baseline rule, recomputed on the warehouse feature set, same test split as the model
train_visible = train_grp[train_grp["avg_position_first_half"] > 0]
tier_ctr_train = (
    train_visible.groupby("position_tier")
    .apply(lambda g: g["clk_first_half"].sum() / g["imp_first_half"].sum())
    .rename("tier_ctr")
)
print("Tier CTR baselines (from train clients only):")
print(tier_ctr_train.round(4))

expected_ctr = test_grp["position_tier"].map(tier_ctr_train)
has_position   = test_grp["avg_position_first_half"] > 0
visible_mask   = test_grp["imp_first_half"] >= 50
underperforms  = has_position & (test_grp["ctr_first_half"] < expected_ctr)
low_active     = test_grp["active_days_first_half"] <= 7
rule_triggered = visible_mask & underperforms & low_active

baseline_score = np.where(rule_triggered, test_grp["imp_first_half"].values, 0).astype(float)
p20_rule = precision_at_k(test_grp[TARGET], baseline_score, 20)
p50_rule = precision_at_k(test_grp[TARGET], baseline_score, 50)
print(f"\nRule baseline (warehouse, test clients): P@20={p20_rule:.3f}  P@50={p50_rule:.3f}")
print(f"Rule fires on {rule_triggered.sum():,} / {len(test_grp):,} test rows ({rule_triggered.mean():.1%})")


## 4. Results (vs baseline) — warehouse, client-grouped test set

All numbers above are computed on the **client-grouped test set** — pages from clients never
seen during training — so the baseline rule, the naive-split model, and the honest model are
all comparable. The cell below assembles the summary table and precision@K chart from the
values computed in Section 3; run it after the cells above to get real numbers.

**Reading it honestly:** the naive random-split P@50 is expected to sit meaningfully above the
client-grouped P@50, the same pattern found on the starter sample (0.92 → 0.56). Report both
numbers rather than the flattering one — the gap itself is evidence about what a random split
was actually measuring.


In [ ]:
results = pd.DataFrame([
    {"Model": "Base rate (always flag)", "P@20": after["base_rate"], "P@50": after["base_rate"], "ROC-AUC": 0.500},
    {"Model": "Rule baseline (warehouse)", "P@20": p20_rule, "P@50": p50_rule, "ROC-AUC": float("nan")},
    {"Model": "Random Forest -- naive random split", "P@20": before["P@20"], "P@50": before["P@50"], "ROC-AUC": before["ROC-AUC"]},
    {"Model": "Random Forest -- client-grouped split (honest)", "P@20": after["P@20"], "P@50": after["P@50"], "ROC-AUC": after["ROC-AUC"]},
])
print(results.to_string(index=False, float_format="{:.3f}".format))

import os
os.makedirs("../outputs", exist_ok=True)
results.to_csv("../outputs/capstone_warehouse_model_comparison.csv", index=False)
print("\nSaved -> work/outputs/capstone_warehouse_model_comparison.csv")


In [ ]:
import matplotlib.pyplot as plt

Ks = [5, 10, 20, 30, 50, 75, 100]
rf_grp = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=10, class_weight="balanced", random_state=SEED, n_jobs=-1)),
]).fit(train_grp[NUMERIC_FEATURES + CATEGORICAL_FEATURES], train_grp[TARGET])
rf_proba_grp = rf_grp.predict_proba(test_grp[NUMERIC_FEATURES + CATEGORICAL_FEATURES])[:, 1]

fig, ax = plt.subplots(figsize=(8, 4))
for name, scores, style in [
    ("Rule baseline", baseline_score, "--s"),
    ("Random Forest (client-grouped)", rf_proba_grp, "-^"),
]:
    curve = [precision_at_k(test_grp[TARGET], scores, k) for k in Ks]
    ax.plot(Ks, curve, style, label=name, linewidth=1.8, markersize=6)
ax.axhline(after["base_rate"], color="grey", linestyle=":", linewidth=1.2, label=f"Base rate ({after['base_rate']:.1%})")
ax.set_xlabel("K (top-K pages reviewed)")
ax.set_ylabel("Precision@K")
ax.set_title("Precision@K -- Warehouse, Rule Baseline vs Random Forest (client-grouped)")
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/capstone_warehouse_precision_at_k.png", dpi=120)
plt.show()
print("Saved -> work/outputs/capstone_warehouse_precision_at_k.png")


## 5. Limitations & honest framing

Carried forward directly from `w03_data_contract.ipynb`'s own "Data limits" section, plus what
this extension adds:

- **The label is a short, noisy proxy window.** A first-half-vs-second-half split inside one
  calendar month can flag a single bad week as "decline" just as easily as a real sustained
  drop, and it can't rule out consolidation (a sibling page absorbing the traffic) or
  seasonality (calendar-driven demand shift) — both need multi-month history this one-month
  slice doesn't have. The real next step for this lane is a genuine future-window label (prior
  90 days predicting the next 30), built across the multi-month panel.
- **Unbalanced panel coverage.** Not every client had GSC history live by March 2026, and rows
  before a client's `ga4_data_start` never carry real engagement data — both were controlled
  for above (`gsc_data_start`/`ga4_data_available IS TRUE` checks), but a single month can't
  say whether March itself was a typical month for any given client.
- **Client-grouped honesty has a cost, and that's the point.** The gap between the naive
  random-split P@50 and the honest client-grouped P@50 (Section 4) means part of the apparent
  skill on a client the model has already seen does not transfer to a client it has never met
  — the same finding as the starter-sample capstone, now confirmed on warehouse data too.
- **Single-month scope, not the full panel.** This notebook uses one mid-panel month
  (`2026-03`) of the ~79M-row warehouse, not the full multi-month history — a deliberate scope
  choice to keep the analysis honest and reproducible, not a full warehouse-scale finding.
- **Cross-sectional, not causal.** Nothing here claims that refreshing a flagged page *will*
  fix it — only that it looks worth reviewing first. No claims are made about Google's
  algorithm, per the internship's public-data rule.


## 6. Ranked recommendations — the action playbook (warehouse)

Same reason-code and action-mapping design as the starter-sample playbook (`w07`), rebuilt on
the warehouse-trained, client-grouped Random Forest above.

**Reason codes (any subset can fire per page):**
- `stale_ctr_underperformer` — the rule from Section 3.2, recomputed here
- `high_decline_risk` — `decline_proba >= 0.6` from the honest, client-grouped model
- `low_engagement_visible` — visible in GSC (`imp_first_half` above the floor) but
  `has_ga4_data == 1` and `engagement_rate_first_half` well below the position tier's median —
  a warehouse-only signal not available in the starter sample
- `stable_top_performer` — `top_3`/`page_1` tier, low decline risk, real first-half traffic —
  a *protect* signal, not a refresh signal

**Action mapping:** identical logic to `w07` — `stable_top_performer` → **protect**; 2+ risk
codes → **refresh_priority**; exactly 1 → **refresh_review**; none → **monitor**.

**Priority score:** `decline_proba × imp_first_half` (capped at the 99th percentile).

**Intended use, human-review checklist, and no-go list:** identical to `w07`'s — this is a
weekly/biweekly triage input for a human reviewer, never an auto-actioning system. No
auto-publishing, no auto-deleting `monitor` rows, no client-facing raw scores, no cross-client
performance ranking off this queue.


In [ ]:
# score the full active set with the warehouse-trained model, apply reason codes
data_scored = model_data.copy()
data_scored["decline_proba"] = rf_grp.predict_proba(data_scored[NUMERIC_FEATURES + CATEGORICAL_FEATURES])[:, 1]

tier_engagement_median = (
    data_scored[data_scored["has_ga4_data"] == 1]
    .groupby("position_tier")["engagement_rate_first_half"].median()
)
expected_engagement = data_scored["position_tier"].map(tier_engagement_median)

expected_ctr_full = data_scored["position_tier"].map(tier_ctr_train)
data_scored["stale_ctr_underperformer"] = (
    (data_scored["imp_first_half"] >= 50) &
    (data_scored["active_days_first_half"] <= 7) &
    (data_scored["ctr_first_half"] < expected_ctr_full)
)
data_scored["high_decline_risk"] = data_scored["decline_proba"] >= 0.6
data_scored["low_engagement_visible"] = (
    (data_scored["has_ga4_data"] == 1) &
    (data_scored["imp_first_half"] >= 50) &
    (data_scored["engagement_rate_first_half"] < expected_engagement * 0.5)
)
data_scored["stable_top_performer"] = (
    data_scored["position_tier"].isin(["top_3", "page_1"]) &
    (data_scored["decline_proba"] < 0.3) &
    (data_scored["imp_first_half"] >= 50)
)

RISK_COLS = ["stale_ctr_underperformer", "high_decline_risk", "low_engagement_visible"]
REASON_COLS = RISK_COLS + ["stable_top_performer"]
data_scored["reason_codes"] = data_scored[REASON_COLS].apply(
    lambda r: "|".join(r.index[r]) if r.any() else "none", axis=1
)

def action_for(row):
    if row["stable_top_performer"]:
        return "protect"
    n_risk = sum(row[c] for c in RISK_COLS)
    if n_risk >= 2:
        return "refresh_priority"
    if n_risk == 1:
        return "refresh_review"
    return "monitor"

data_scored["action"] = data_scored.apply(action_for, axis=1)
imp_cap = data_scored["imp_first_half"].quantile(0.99)
data_scored["priority_score"] = (data_scored["decline_proba"] * data_scored["imp_first_half"].clip(upper=imp_cap)).round(1)

print("Action distribution:")
print(data_scored["action"].value_counts())

queue = data_scored[data_scored["action"].isin(["refresh_priority", "refresh_review"])].sort_values("priority_score", ascending=False)
show_cols = ["content_hash_id", "action", "reason_codes", "position_tier", "decline_proba", "priority_score"]
print("\nTop 10 refresh_priority / refresh_review rows:")
print(queue[show_cols].head(10).to_string(index=False))


## 7. Artifacts the paper embeds

Committed outputs specific to the warehouse extension (kept separate from the starter-sample
capstone artifacts so the paper can present both honestly):

| Artifact | Committed? |
|---|---|
| `work/outputs/capstone_warehouse_model_comparison.csv` | yes — summary numbers |
| `work/outputs/capstone_warehouse_precision_at_k.png` | yes — chart |
| `work/outputs/capstone_warehouse_headline.json` | yes — the receipts, single source of truth |
| row-level queue (if exported) | no — gitignored, regenerate on demand |


In [ ]:
import json

headline = {
    "lane": "Lane 2 -- Refresh / Content Opportunity Scoring",
    "data_source": "FlyRank/internship-warehouse (Hugging Face), fact_content_daily_performance, month=2026-03",
    "unit_of_analysis": "one (client_hash_id, content_hash_id) pair, aggregated over the first half of March",
    "label": "declining_second_half -- proxy: did GSC impressions fall from first half to second half of the month",
    "validation_design": "client-grouped holdout (~20% of clients held out entirely)",
    "split_design_finding": {
        "random_split_P50": before["P@50"],
        "client_grouped_P50": after["P@50"],
        "gap": round(before["P@50"] - after["P@50"], 3),
    },
    "final_model": {
        "type": "RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced')",
        "P50": after["P@50"], "P20": after["P@20"], "ROC_AUC": after["ROC-AUC"], "base_rate": after["base_rate"],
    },
    "baseline_rule_P50": p50_rule,
    "relationship_to_starter_sample_capstone": (
        "Same lane, same overall methodology (baseline -> model -> honest grouped validation "
        "-> action playbook) as the starter-sample capstone. This extension runs the pipeline "
        "on the full warehouse using the contract from w03_data_contract.ipynb, and reports "
        "the client-grouped honesty gap measured directly on warehouse data."
    ),
}

with open("../outputs/capstone_warehouse_headline.json", "w") as f:
    json.dump(headline, f, indent=2, default=float)

print(json.dumps(headline, indent=2, default=float))
print("\nWrote -> work/outputs/capstone_warehouse_headline.json")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] Column names verified against `docs/data-dictionary.md` and adjusted if they differ
- [ ] The notebook runs top to bottom with no errors, with a valid HF token, Runtime → Run all
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] The baseline, naive-split model, and honest grouped-split model all appear on the same, comparable test set
- [ ] Limitations section names the proxy label, panel coverage, and the grouped-split cost explicitly
- [ ] Ranked recommendations include reason codes, an action mapping, and the same no-go list as `w07`
- [ ] `work/outputs/capstone_warehouse_headline.json` regenerated and committed
- [ ] Deployed paper URL added to `submission/paper_url.txt`
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
